In [ ]:
import re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [ ]:
docs = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog are friends",
    "machine learning is fun",
]

In [ ]:
tok = lambda t: re.findall(r"(?u)\b\w\w+\b", t.lower())
vocab = sorted({w for d in docs for w in tok(d)})
N = len(docs)

tf = np.array([[tok(d).count(w) for w in vocab] for d in docs], dtype=float)
df_ = (tf > 0).sum(axis=0)
idf = np.log((1 + N) / (1 + df_)) + 1
tfidf = tf * idf
tfidf = tfidf / np.linalg.norm(tfidf, axis=1, keepdims=True)      # L2 normalisation

idx = [f"D{i+1}" for i in range(N)]
df_tfidf = pd.DataFrame(tfidf.round(3), columns=vocab, index=idx)
df_tfidf

In [ ]:
sk = TfidfVectorizer().fit(docs)
sk_mat = sk.transform(docs).toarray()
print("Manual == sklearn TF-IDF :", np.allclose(tfidf, sk_mat))
print("\nIDF values:")
print(pd.Series(idf, index=vocab).round(3).sort_values())

In [ ]:
bow = CountVectorizer().fit_transform(docs).toarray()
cmp = pd.DataFrame({"BoW count": bow[0], "TF-IDF": tfidf[0].round(3)}, index=vocab)
display(cmp)

cmp.plot(kind="bar", subplots=True, figsize=(9, 5), legend=False, title=["BoW count (D1)", "TF-IDF weight (D1)"])
plt.tight_layout(); plt.show()